# Base 0 (Zero-Shot) Inference & Tracking Orchestrator

This notebook performs sparse repository cloning, installs dependencies in editable mode, runs unit tests, and executes the Base 0 inference and tracking runner in an isolated process to prevent kernel restart prompts.

In [ ]:
import os
from pathlib import Path

REPO_NAME = 'ia_article'
REPO_URL = 'https://github.com/unsa-semester-2026-A/ia_article.git'
BRANCH_NAME = 'feat/19-base0-evaluation'

# --- Environment Detection: Kaggle vs Colab ---
BASE_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
%cd {BASE_DIR}

REPO_PATH = Path(BASE_DIR) / REPO_NAME

# 1. Sparse clone repository from feature branch
if not REPO_PATH.exists():
    print(f"Cloning {REPO_NAME} (branch {BRANCH_NAME})...")
    !git clone -q --depth 1 --branch {BRANCH_NAME} --filter=blob:none --sparse {REPO_URL}
    %cd {REPO_NAME}
    !git sparse-checkout set experiments
    %cd experiments
else:
    print(f"Updating existing {REPO_NAME} repository...")
    %cd {REPO_NAME}
    !git checkout {BRANCH_NAME}
    !git pull -q origin {BRANCH_NAME}
    %cd experiments

# 2. Verify working directory
current_dir = Path(os.getcwd())
if current_dir.name != 'experiments':
    raise RuntimeError(f"Directory navigation failed. Current path: {current_dir}")

# 3. Install package in editable mode
print("Installing package in editable mode with [cloud] dependencies...")
%pip install -q -e .[cloud]

## 2. Run Unit Tests

Execute all co-located unit tests (IOManager, BaseInferencePipeline, and Base0Runner) to ensure system stability.

In [ ]:
!pytest src/

## 3. Execute Base 0 Inference & Tracking Runner

Runs the `run_base_0` module in a separate process (`!python3 -m ...`). This performs health checks, dataset discovery, ByteTrack OBB tracking, Google Drive uploads, and automatic session shutdown.

In [ ]:
!python3 -m src.inference.runners.run_base_0